# core

> Reusable housing price machine learning pipeline for nbdev projects.


In [ ]:
#| default_exp core


In [ ]:
#| hide
from nbdev.showdoc import *


## Overview

This module packages the housing price workflow from `script/house-pricing-ml-pipeline.ipynb` into a reusable Python API. It keeps the original feature-engine preprocessing pipeline, adds the missing custom transformers, and exposes helpers for loading data, training, evaluating, predicting, submitting, and persisting models.


## Imports and configuration

The constants below mirror the selected features and mappings used by the original pipeline notebook. The default CSV paths point at the local `script/` folder in this repository.


In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import joblib
import numpy as np
import pandas as pd
from feature_engine.encoding import OrdinalEncoder, RareLabelEncoder
from feature_engine.imputation import AddMissingIndicator, CategoricalImputer, MeanMedianImputer
from feature_engine.selection import DropFeatures
from feature_engine.transformation import LogTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Binarizer, MinMaxScaler

DEFAULT_RANDOM_STATE = 0
DEFAULT_TEST_SIZE = 0.1
TARGET = "SalePrice"
ID_COLUMN = "Id"


def _find_project_root() -> Path:
    "Return the nearest parent that looks like this nbdev project."
    starts: list[Path] = [Path.cwd()]
    if "__file__" in globals():
        starts.insert(0, Path(__file__).resolve())

    for start in starts:
        base = start if start.is_dir() else start.parent
        for candidate in (base, *base.parents):
            if (candidate / "nbs" / "nbdev.yml").exists() or (candidate / "script" / "train.csv").exists():
                return candidate
    return Path.cwd()


PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "script"
DEFAULT_TRAIN_PATH = DATA_DIR / "train.csv"
DEFAULT_TEST_PATH = DATA_DIR / "test.csv"

CATEGORICAL_VARS_WITH_NA_FREQUENT = [
    "BsmtQual",
    "BsmtExposure",
    "BsmtFinType1",
    "GarageFinish",
]
CATEGORICAL_VARS_WITH_NA_MISSING = ["FireplaceQu"]
NUMERICAL_VARS_WITH_NA = ["LotFrontage"]

TEMPORAL_VARS = ["YearRemodAdd"]
REF_VAR = "YrSold"
DROP_FEATURES = ["YrSold"]
NUMERICALS_LOG_VARS = ["LotFrontage", "1stFlrSF", "GrLivArea"]
BINARIZE_VARS = ["ScreenPorch"]

QUAL_VARS = ["ExterQual", "BsmtQual", "HeatingQC", "KitchenQual", "FireplaceQu"]
EXPOSURE_VARS = ["BsmtExposure"]
FINISH_VARS = ["BsmtFinType1"]
GARAGE_VARS = ["GarageFinish"]
FENCE_VARS = ["Fence"]

CATEGORICAL_VARS = [
    "MSSubClass",
    "MSZoning",
    "LotShape",
    "LandContour",
    "LotConfig",
    "Neighborhood",
    "RoofStyle",
    "Exterior1st",
    "Foundation",
    "CentralAir",
    "Functional",
    "PavedDrive",
    "SaleCondition",
]

QUAL_MAPPINGS = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5, "Missing": 0, "NA": 0}
EXPOSURE_MAPPINGS = {"No": 1, "Mn": 2, "Av": 3, "Gd": 4}
FINISH_MAPPINGS = {"Missing": 0, "NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
GARAGE_MAPPINGS = {"Missing": 0, "NA": 0, "Unf": 1, "RFn": 2, "Fin": 3}

FEATURES = [
    "MSSubClass",
    "MSZoning",
    "LotFrontage",
    "LotShape",
    "LandContour",
    "LotConfig",
    "Neighborhood",
    "OverallQual",
    "OverallCond",
    "YearRemodAdd",
    "RoofStyle",
    "Exterior1st",
    "ExterQual",
    "Foundation",
    "BsmtQual",
    "BsmtExposure",
    "BsmtFinType1",
    "HeatingQC",
    "CentralAir",
    "1stFlrSF",
    "2ndFlrSF",
    "GrLivArea",
    "BsmtFullBath",
    "HalfBath",
    "KitchenQual",
    "TotRmsAbvGrd",
    "Functional",
    "Fireplaces",
    "FireplaceQu",
    "GarageFinish",
    "GarageCars",
    "GarageArea",
    "PavedDrive",
    "WoodDeckSF",
    "ScreenPorch",
    "SaleCondition",
    "YrSold",
]

PIPELINE_HANDLED_MISSING_VARS = (
    CATEGORICAL_VARS_WITH_NA_FREQUENT + CATEGORICAL_VARS_WITH_NA_MISSING + NUMERICAL_VARS_WITH_NA
)
HandleMissing = Literal["drop", "raise"]


## Custom transformers

The original script imports these from a `preprocessors` module. Exporting them here makes the pipeline self-contained and serializable as part of this package.


In [ ]:
#| export
def _as_list(values: str | list[str] | tuple[str, ...]) -> list[str]:
    if isinstance(values, str):
        return [values]
    return list(values)


def _check_required_columns(data: pd.DataFrame, required: list[str], label: str = "data") -> None:
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    """Calculate elapsed years between a reference year and temporal variables."""

    def __init__(self, variables: str | list[str], reference_variable: str):
        self.variables = variables
        self.reference_variable = reference_variable

    def fit(self, X: pd.DataFrame, y: Any = None):
        variables = _as_list(self.variables)
        _check_required_columns(X, variables + [self.reference_variable], "X")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        variables = _as_list(self.variables)
        _check_required_columns(X, variables + [self.reference_variable], "X")
        for variable in variables:
            X[variable] = X[self.reference_variable] - X[variable]
        return X


class Mapper(BaseEstimator, TransformerMixin):
    """Map ordered categorical labels to numeric values."""

    def __init__(self, variables: str | list[str], mappings: dict[Any, Any], default_value: Any | None = None):
        self.variables = variables
        self.mappings = mappings
        self.default_value = default_value

    def fit(self, X: pd.DataFrame, y: Any = None):
        _check_required_columns(X, _as_list(self.variables), "X")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        _check_required_columns(X, _as_list(self.variables), "X")
        for variable in _as_list(self.variables):
            mapped = X[variable].map(self.mappings)
            if self.default_value is not None:
                mapped = mapped.fillna(self.default_value)
            X[variable] = mapped
        return X


In [ ]:
temporal_sample = pd.DataFrame({"YrSold": [2010], "YearRemodAdd": [2000]})
temporal_result = TemporalVariableTransformer(["YearRemodAdd"], "YrSold").fit_transform(temporal_sample)
assert temporal_result.loc[0, "YearRemodAdd"] == 10

mapping_sample = pd.DataFrame({"KitchenQual": ["TA", "Gd", "Ex"]})
mapping_result = Mapper(["KitchenQual"], QUAL_MAPPINGS).fit_transform(mapping_sample)
assert mapping_result["KitchenQual"].tolist() == [3, 4, 5]


## Data preparation

These helpers load the CSV files, cast `MSSubClass` as categorical, select the modeled feature set, and split the target on the log scale used by the model.


In [ ]:
#| export
def load_housing_data(path: str | Path | None = None) -> pd.DataFrame:
    """Load a housing CSV file, defaulting to the repository training data."""
    csv_path = DEFAULT_TRAIN_PATH if path is None else Path(path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Housing data file not found: {csv_path}")
    return pd.read_csv(csv_path)


def _cast_categorical_inputs(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    if "MSSubClass" in data.columns:
        data["MSSubClass"] = data["MSSubClass"].astype(str)
    return data


def _find_unhandled_missing(data: pd.DataFrame) -> list[str]:
    return [
        variable
        for variable in FEATURES
        if variable not in PIPELINE_HANDLED_MISSING_VARS and data[variable].isna().sum() > 0
    ]


def _prepare_features(data: pd.DataFrame, drop_id: bool = True) -> pd.DataFrame:
    data = _cast_categorical_inputs(data)
    if drop_id and ID_COLUMN in data.columns:
        data = data.drop(columns=ID_COLUMN)
    _check_required_columns(data, FEATURES, "data")
    return data[FEATURES].copy()


def split_housing_data(
    data: pd.DataFrame,
    test_size: float = DEFAULT_TEST_SIZE,
    random_state: int = DEFAULT_RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Split raw housing data into train/validation features and log target values."""
    _check_required_columns(data, [TARGET], "data")
    if (data[TARGET] <= 0).any():
        raise ValueError(f"{TARGET} must contain only positive values for log transformation.")

    X = _prepare_features(data, drop_id=True)
    y = np.log(data[TARGET]).rename(TARGET)
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


In [ ]:
train_data = load_housing_data()
test_data = load_housing_data(DEFAULT_TEST_PATH)

assert train_data.shape == (1460, 81)
assert test_data.shape == (1459, 80)
assert set(FEATURES).issubset(train_data.columns)
assert set(FEATURES).issubset(test_data.columns)


## Pipeline and training

`build_price_pipeline` recreates the source feature-engine pipeline. `train_price_model` returns a compact result object so downstream users can inspect the trained model, data splits, and metrics without retraining.


In [ ]:
#| export
@dataclass
class TrainingResult:
    """Container returned by `train_price_model`."""

    model: Pipeline
    X_train: pd.DataFrame
    X_valid: pd.DataFrame
    y_train: pd.Series
    y_valid: pd.Series
    metrics: dict[str, float]


def build_price_pipeline() -> Pipeline:
    """Build the end-to-end housing price pipeline from the original script notebook."""
    return Pipeline(
        [
            (
                "missing_imputation",
                CategoricalImputer(
                    imputation_method="missing",
                    variables=CATEGORICAL_VARS_WITH_NA_MISSING,
                ),
            ),
            (
                "frequent_imputation",
                CategoricalImputer(
                    imputation_method="frequent",
                    variables=CATEGORICAL_VARS_WITH_NA_FREQUENT,
                ),
            ),
            ("missing_indicator", AddMissingIndicator(variables=NUMERICAL_VARS_WITH_NA)),
            (
                "mean_imputation",
                MeanMedianImputer(
                    imputation_method="mean",
                    variables=NUMERICAL_VARS_WITH_NA,
                ),
            ),
            ("elapsed_time", TemporalVariableTransformer(variables=TEMPORAL_VARS, reference_variable=REF_VAR)),
            ("drop_features", DropFeatures(features_to_drop=DROP_FEATURES)),
            ("log", LogTransformer(variables=NUMERICALS_LOG_VARS)),
            (
                "binarizer",
                SklearnTransformerWrapper(transformer=Binarizer(threshold=0), variables=BINARIZE_VARS),
            ),
            ("mapper_qual", Mapper(variables=QUAL_VARS, mappings=QUAL_MAPPINGS)),
            ("mapper_exposure", Mapper(variables=EXPOSURE_VARS, mappings=EXPOSURE_MAPPINGS)),
            ("mapper_finish", Mapper(variables=FINISH_VARS, mappings=FINISH_MAPPINGS)),
            ("mapper_garage", Mapper(variables=GARAGE_VARS, mappings=GARAGE_MAPPINGS)),
            (
                "rare_label_encoder",
                RareLabelEncoder(tol=0.01, n_categories=1, variables=CATEGORICAL_VARS),
            ),
            (
                "categorical_encoder",
                OrdinalEncoder(encoding_method="ordered", variables=CATEGORICAL_VARS),
            ),
            ("scaler", MinMaxScaler()),
            ("Lasso", Lasso(alpha=0.001, random_state=DEFAULT_RANDOM_STATE)),
        ]
    )


def evaluate_price_model(
    model: Pipeline,
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_train: pd.Series,
    y_valid: pd.Series,
) -> dict[str, float]:
    """Evaluate a fitted model on original dollar-scale prices."""
    train_predictions = model.predict(X_train)
    valid_predictions = model.predict(X_valid)

    train_actual = np.exp(y_train)
    valid_actual = np.exp(y_valid)
    train_predicted = np.exp(train_predictions)
    valid_predicted = np.exp(valid_predictions)

    return {
        "train_mse": float(mean_squared_error(train_actual, train_predicted)),
        "train_rmse": float(root_mean_squared_error(train_actual, train_predicted)),
        "train_r2": float(r2_score(train_actual, train_predicted)),
        "valid_mse": float(mean_squared_error(valid_actual, valid_predicted)),
        "valid_rmse": float(root_mean_squared_error(valid_actual, valid_predicted)),
        "valid_r2": float(r2_score(valid_actual, valid_predicted)),
    }


def train_price_model(
    train_path: str | Path | None = None,
    test_size: float = DEFAULT_TEST_SIZE,
    random_state: int = DEFAULT_RANDOM_STATE,
) -> TrainingResult:
    """Train the default price model and return the fitted model, splits, and metrics."""
    data = load_housing_data(train_path)
    X_train, X_valid, y_train, y_valid = split_housing_data(
        data,
        test_size=test_size,
        random_state=random_state,
    )
    model = build_price_pipeline()
    model.fit(X_train, y_train)
    metrics = evaluate_price_model(model, X_train, X_valid, y_train, y_valid)
    return TrainingResult(model, X_train, X_valid, y_train, y_valid, metrics)


In [ ]:
pipeline = build_price_pipeline()
assert pipeline.steps[-1][0] == "Lasso"
assert isinstance(pipeline, Pipeline)


In [ ]:
result = train_price_model()
result.metrics


In [ ]:
assert result.metrics["valid_r2"] > 0.80
assert result.metrics["valid_rmse"] > 0
assert result.X_train.shape[1] == len(FEATURES)
assert result.X_valid.shape[1] == len(FEATURES)


## Prediction and submission

The pipeline learns missing-value handling for the variables configured above. The Kaggle test set also has a few missing values in variables the source pipeline does not impute, so prediction defaults to dropping those rows, matching the original script notebook.


In [ ]:
#| export
def _prepare_prediction_features(
    data: pd.DataFrame,
    drop_id: bool = True,
    handle_missing: HandleMissing = "drop",
) -> pd.DataFrame:
    if handle_missing not in {"drop", "raise"}:
        raise ValueError("handle_missing must be either 'drop' or 'raise'.")

    X = _prepare_features(data, drop_id=drop_id)
    unhandled_missing = _find_unhandled_missing(X)
    if unhandled_missing and handle_missing == "raise":
        raise ValueError(f"Prediction data has missing values not handled by the pipeline: {unhandled_missing}")
    if unhandled_missing:
        X = X.dropna(subset=unhandled_missing)
    return X


def predict_prices(
    model: Pipeline,
    data: pd.DataFrame,
    drop_id: bool = True,
    handle_missing: HandleMissing = "drop",
) -> pd.Series:
    """Predict house prices on the original dollar scale."""
    X = _prepare_prediction_features(data, drop_id=drop_id, handle_missing=handle_missing)
    predictions = np.exp(model.predict(X))
    return pd.Series(predictions, index=X.index, name=TARGET)


def make_submission(
    model: Pipeline,
    test_path: str | Path | None = None,
    id_column: str = ID_COLUMN,
    handle_missing: HandleMissing = "drop",
) -> pd.DataFrame:
    """Create a Kaggle-style submission DataFrame with `Id` and `SalePrice` columns."""
    data = load_housing_data(DEFAULT_TEST_PATH if test_path is None else test_path)
    _check_required_columns(data, [id_column], "test data")
    predictions = predict_prices(model, data, drop_id=True, handle_missing=handle_missing)
    return pd.DataFrame(
        {
            id_column: data.loc[predictions.index, id_column].to_numpy(),
            TARGET: predictions.to_numpy(),
        }
    )


In [ ]:
submission = make_submission(result.model)
submission.head()


In [ ]:
assert list(submission.columns) == [ID_COLUMN, TARGET]
assert len(submission) == 1449
assert submission[TARGET].notna().all()
assert (submission[TARGET] > 0).all()

sample_prices = predict_prices(result.model, test_data.head(20))
assert len(sample_prices) == 20
assert (sample_prices > 0).all()


## Persistence

Use these helpers to save the trained pipeline and load it again for later scoring.


In [ ]:
#| export
def save_model(model: Pipeline, path: str | Path) -> Path:
    """Persist a fitted model with joblib and return the output path."""
    model_path = Path(path)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_path)
    return model_path


def load_model(path: str | Path) -> Pipeline:
    """Load a model saved with `save_model`."""
    model_path = Path(path)
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    return joblib.load(model_path)


In [ ]:
model_path = save_model(result.model, PROJECT_ROOT / "_proc" / "price_pipe.joblib")
loaded_model = load_model(model_path)
loaded_prices = predict_prices(loaded_model, test_data.head(5))
assert len(loaded_prices) == 5
assert (loaded_prices > 0).all()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
